# 16. Conditioning and DiT modulation — FiLM, cross-attention, DiT-B/2, and SD3 MMDiT

Only tensor widths, image/token counts, batch size, and training budget are reduced. The demonstrated model depths and branch topology are kept.

- FiLM and cross-attention are shown as primitive conditioning operators.
- **DiT-B/2** keeps patch size 2, 12 Transformer blocks, 12 attention heads, adaLN-Zero on every block, class/timestep conditioning, and the modulated final layer.
- **Stable Diffusion 3 Medium MMDiT** keeps patch size 2, 24 joint blocks, 24 attention heads, separate image/context parameters, joint attention, and the final `context_pre_only` block.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(7)
torch.set_num_threads(min(2, torch.get_num_threads()))
device = torch.device("cpu")
print("device:", device)


## 1. FiLM and cross-attention primitives


In [ ]:
features = torch.randn(2, 5, 8, device=device)
condition = torch.randn(2, 4, device=device)
film_projection = nn.Linear(4, 16).to(device)
scale, shift = film_projection(condition).chunk(2, dim=-1)
film_output = features * (1 + scale[:, None]) + shift[:, None]

image_tokens = torch.randn(2, 6, 12, device=device)
text_tokens = torch.randn(2, 4, 12, device=device)
query_projection = nn.Linear(12, 12, bias=False).to(device)
key_projection = nn.Linear(12, 12, bias=False).to(device)
value_projection = nn.Linear(12, 12, bias=False).to(device)

q = query_projection(image_tokens).view(2, 6, 3, 4).transpose(1, 2)
k = key_projection(text_tokens).view(2, 4, 3, 4).transpose(1, 2)
v = value_projection(text_tokens).view(2, 4, 3, 4).transpose(1, 2)
cross_attention = F.scaled_dot_product_attention(q, k, v)

print("FiLM:", film_output.shape)
print("cross-attention:", cross_attention.shape)


## 2. DiT-B/2 — full 12-block adaLN-Zero stack

The hidden width is reduced, but the DiT-B/2 depth, number of heads, patch size, conditioning path, residual gates and final adaptive LayerNorm path remain.


In [ ]:
def sinusoidal_timestep_embedding(timestep, dim, max_period=10000):
    half = dim // 2
    frequencies = torch.exp(
        -math.log(max_period)
        * torch.arange(half, device=timestep.device, dtype=torch.float32)
        / half
    )
    arguments = timestep.float()[:, None] * frequencies[None]
    embedding = torch.cat([arguments.cos(), arguments.sin()], dim=-1)
    if dim % 2:
        embedding = torch.cat(
            [embedding, torch.zeros_like(embedding[:, :1])],
            dim=-1,
        )
    return embedding


def modulate(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]


class DiTSelfAttention(nn.Module):
    def __init__(self, hidden_dim=48, heads=12):
        super().__init__()
        assert hidden_dim % heads == 0
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim, bias=True)
        self.out = nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        batch, length, hidden_dim = x.shape
        qkv = self.qkv(x).view(
            batch,
            length,
            3,
            self.heads,
            self.head_dim,
        )
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        attended = F.scaled_dot_product_attention(q, k, v)
        attended = attended.transpose(1, 2).contiguous().view(
            batch,
            length,
            hidden_dim,
        )
        return self.out(attended)


class DiTBlock(nn.Module):
    def __init__(self, hidden_dim=48, heads=12):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=1e-6)
        self.attention = DiTSelfAttention(hidden_dim, heads)
        self.norm2 = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )
        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 6 * hidden_dim),
        )
        nn.init.zeros_(self.modulation[-1].weight)
        nn.init.zeros_(self.modulation[-1].bias)

    def forward(self, x, condition):
        (
            shift_attention,
            scale_attention,
            gate_attention,
            shift_mlp,
            scale_mlp,
            gate_mlp,
        ) = self.modulation(condition).chunk(6, dim=-1)

        attention_input = modulate(
            self.norm1(x),
            shift_attention,
            scale_attention,
        )
        x = x + gate_attention[:, None] * self.attention(attention_input)

        mlp_input = modulate(
            self.norm2(x),
            shift_mlp,
            scale_mlp,
        )
        return x + gate_mlp[:, None] * self.mlp(mlp_input)


class DiTFinalLayer(nn.Module):
    def __init__(self, hidden_dim, patch_size, output_channels):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False, eps=1e-6)
        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 2 * hidden_dim),
        )
        self.linear = nn.Linear(
            hidden_dim,
            patch_size * patch_size * output_channels,
        )
        nn.init.zeros_(self.modulation[-1].weight)
        nn.init.zeros_(self.modulation[-1].bias)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x, condition):
        shift, scale = self.modulation(condition).chunk(2, dim=-1)
        return self.linear(modulate(self.norm(x), shift, scale))


class SmallWidthDiTB2(nn.Module):
    def __init__(
        self,
        image_size=8,
        patch_size=2,
        input_channels=4,
        hidden_dim=48,
        heads=12,
        depth=12,
        classes=10,
        class_dropout=0.1,
        learn_sigma=True,
    ):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.input_channels = input_channels
        self.output_channels = input_channels * 2 if learn_sigma else input_channels
        self.class_dropout = class_dropout
        self.classes = classes

        self.patch = nn.Conv2d(
            input_channels,
            hidden_dim,
            patch_size,
            stride=patch_size,
        )
        token_count = (image_size // patch_size) ** 2
        self.position = nn.Parameter(torch.randn(1, token_count, hidden_dim) * 0.02)

        self.time_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.label_embedding = nn.Embedding(classes + 1, hidden_dim)
        self.blocks = nn.ModuleList(
            [DiTBlock(hidden_dim, heads) for _ in range(depth)]
        )
        self.final = DiTFinalLayer(
            hidden_dim,
            patch_size,
            self.output_channels,
        )

    def condition(self, timestep, labels, training=True):
        time_embedding = sinusoidal_timestep_embedding(
            timestep,
            self.position.size(-1),
        )
        time_embedding = self.time_mlp(time_embedding)

        labels = labels.clone()
        if training and self.class_dropout > 0:
            drop = torch.rand(labels.shape, device=labels.device) < self.class_dropout
            labels[drop] = self.classes
        return time_embedding + self.label_embedding(labels)

    def forward(self, image, timestep, labels):
        hidden = self.patch(image).flatten(2).transpose(1, 2)
        hidden = hidden + self.position[:, : hidden.size(1)]
        condition = self.condition(timestep, labels, self.training)

        for block in self.blocks:
            hidden = block(hidden, condition)

        patches = self.final(hidden, condition).transpose(1, 2)
        output = F.fold(
            patches,
            output_size=(image.size(-2), image.size(-1)),
            kernel_size=self.patch_size,
            stride=self.patch_size,
        )
        return output


dit = SmallWidthDiTB2().to(device)
assert dit.patch_size == 2
assert len(dit.blocks) == 12
assert all(block.attention.heads == 12 for block in dit.blocks)
assert dit.output_channels == 2 * dit.input_channels

latent = torch.randn(1, 4, 8, 8, device=device)
timestep = torch.tensor([500], device=device)
labels = torch.tensor([3], device=device)
dit_output = dit(latent, timestep, labels)
dit_output.square().mean().backward()
print("DiT-B/2 blocks/heads:", len(dit.blocks), dit.blocks[0].attention.heads)
print("DiT output:", dit_output.shape)


## 3. Stable Diffusion 3 Medium — full 24-block MMDiT transformer

SD3 Medium uses 24 joint Transformer blocks and 24 attention heads. Image and context streams have independent normalization, modulation, Q/K/V, output projection and MLP parameters, while the attention calculation is joint. The final block is `context_pre_only`: context still supplies K/V to joint attention but is not updated afterward.


In [ ]:
class AdaLayerNormZero(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(dim, 6 * dim),
        )

    def forward(self, x, condition):
        (
            shift_attention,
            scale_attention,
            gate_attention,
            shift_mlp,
            scale_mlp,
            gate_mlp,
        ) = self.modulation(condition).chunk(6, dim=-1)
        normalized = modulate(
            self.norm(x),
            shift_attention,
            scale_attention,
        )
        return (
            normalized,
            gate_attention,
            shift_mlp,
            scale_mlp,
            gate_mlp,
        )


class AdaLayerNormContinuous(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(dim, 2 * dim),
        )

    def forward(self, x, condition):
        shift, scale = self.modulation(condition).chunk(2, dim=-1)
        return modulate(self.norm(x), shift, scale)


class MMDiTStream(nn.Module):
    def __init__(self, dim=48, heads=24, context_pre_only=False):
        super().__init__()
        assert dim % heads == 0
        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.context_pre_only = context_pre_only

        if context_pre_only:
            self.norm1 = AdaLayerNormContinuous(dim)
            self.norm2 = None
            self.mlp = None
        else:
            self.norm1 = AdaLayerNormZero(dim)
            self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
            self.mlp = nn.Sequential(
                nn.Linear(dim, 4 * dim),
                nn.GELU(approximate="tanh"),
                nn.Linear(4 * dim, dim),
            )

        self.qkv = nn.Linear(dim, 3 * dim, bias=True)
        self.out = nn.Linear(dim, dim, bias=True)

    def project_qkv(self, normalized):
        batch, length, _ = normalized.shape
        qkv = self.qkv(normalized).view(
            batch,
            length,
            3,
            self.heads,
            self.head_dim,
        ).permute(2, 0, 3, 1, 4)
        return qkv.unbind(0)


class SD3JointTransformerBlock(nn.Module):
    def __init__(self, dim=48, heads=24, context_pre_only=False):
        super().__init__()
        self.context_pre_only = context_pre_only
        self.image = MMDiTStream(dim, heads, context_pre_only=False)
        self.context = MMDiTStream(
            dim,
            heads,
            context_pre_only=context_pre_only,
        )

    @staticmethod
    def merge_heads(x):
        return x.transpose(1, 2).contiguous().flatten(2)

    def forward(self, image, context, condition):
        (
            image_normalized,
            image_gate_attention,
            image_shift_mlp,
            image_scale_mlp,
            image_gate_mlp,
        ) = self.image.norm1(image, condition)

        if self.context_pre_only:
            context_normalized = self.context.norm1(context, condition)
        else:
            (
                context_normalized,
                context_gate_attention,
                context_shift_mlp,
                context_scale_mlp,
                context_gate_mlp,
            ) = self.context.norm1(context, condition)

        image_q, image_k, image_v = self.image.project_qkv(image_normalized)
        context_q, context_k, context_v = self.context.project_qkv(context_normalized)

        joint_q = torch.cat([image_q, context_q], dim=2)
        joint_k = torch.cat([image_k, context_k], dim=2)
        joint_v = torch.cat([image_v, context_v], dim=2)
        joint_output = F.scaled_dot_product_attention(
            joint_q,
            joint_k,
            joint_v,
        )

        image_length = image.size(1)
        image_attention = self.merge_heads(joint_output[:, :, :image_length])
        context_attention = self.merge_heads(joint_output[:, :, image_length:])

        image = image + image_gate_attention[:, None] * self.image.out(image_attention)
        image_mlp_input = modulate(
            self.image.norm2(image),
            image_shift_mlp,
            image_scale_mlp,
        )
        image = image + image_gate_mlp[:, None] * self.image.mlp(image_mlp_input)

        if self.context_pre_only:
            context = None
        else:
            context = context + context_gate_attention[:, None] * self.context.out(
                context_attention
            )
            context_mlp_input = modulate(
                self.context.norm2(context),
                context_shift_mlp,
                context_scale_mlp,
            )
            context = context + context_gate_mlp[:, None] * self.context.mlp(
                context_mlp_input
            )

        return context, image


class SmallWidthSD3MediumTransformer(nn.Module):
    def __init__(
        self,
        input_channels=4,
        hidden_dim=48,
        heads=24,
        depth=24,
        patch_size=2,
        context_input_dim=32,
        pooled_dim=16,
        sample_size=8,
    ):
        super().__init__()
        self.patch_size = patch_size
        self.input_channels = input_channels
        self.hidden_dim = hidden_dim

        self.patch = nn.Conv2d(
            input_channels,
            hidden_dim,
            patch_size,
            stride=patch_size,
        )
        token_count = (sample_size // patch_size) ** 2
        self.position = nn.Parameter(torch.randn(1, token_count, hidden_dim) * 0.02)
        self.context_projection = nn.Linear(context_input_dim, hidden_dim)

        self.time_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.pooled_projection = nn.Sequential(
            nn.Linear(pooled_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.blocks = nn.ModuleList(
            [
                SD3JointTransformerBlock(
                    hidden_dim,
                    heads,
                    context_pre_only=layer_index == depth - 1,
                )
                for layer_index in range(depth)
            ]
        )
        self.final_norm = AdaLayerNormContinuous(hidden_dim)
        self.output = nn.Linear(
            hidden_dim,
            patch_size * patch_size * input_channels,
        )

    def forward(self, latent, timestep, context, pooled):
        image = self.patch(latent).flatten(2).transpose(1, 2)
        image = image + self.position[:, : image.size(1)]
        context = self.context_projection(context)

        time_embedding = sinusoidal_timestep_embedding(timestep, self.hidden_dim)
        condition = self.time_mlp(time_embedding) + self.pooled_projection(pooled)

        for block in self.blocks:
            context, image = block(image, context, condition)

        image = self.final_norm(image, condition)
        patches = self.output(image).transpose(1, 2)
        return F.fold(
            patches,
            output_size=(latent.size(-2), latent.size(-1)),
            kernel_size=self.patch_size,
            stride=self.patch_size,
        )


sd3 = SmallWidthSD3MediumTransformer().to(device)
assert sd3.patch_size == 2
assert len(sd3.blocks) == 24
assert all(block.image.heads == 24 for block in sd3.blocks)
assert not any(block.context_pre_only for block in sd3.blocks[:-1])
assert sd3.blocks[-1].context_pre_only

sd3_latent = torch.randn(1, 4, 8, 8, device=device)
sd3_timestep = torch.tensor([500], device=device)
sd3_context = torch.randn(1, 4, 32, device=device)
sd3_pooled = torch.randn(1, 16, device=device)
sd3_output = sd3(sd3_latent, sd3_timestep, sd3_context, sd3_pooled)
sd3_output.square().mean().backward()

print("SD3 joint blocks/heads:", len(sd3.blocks), sd3.blocks[0].image.heads)
print("final context_pre_only:", sd3.blocks[-1].context_pre_only)
print("SD3 output:", sd3_output.shape)


## Structural checklist

The assertions explicitly check DiT-B/2 `12 blocks / 12 heads / patch=2 / learned-sigma output` and SD3 Medium `24 joint blocks / 24 heads / patch=2 / last context_pre_only`. Widths are small, but neither model depth is reduced.
